# Rx PD Model - With Advanced Feature Engineering
## Rolling Time-Series Features + Temporal Features

In [ ]:
# Install libraries
!pip install -q --upgrade pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from datetime import timedelta

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, precision_score, recall_score, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print('✓ Libraries imported')

# Step 1: Extract and Load Raw Data

In [ ]:
# Extract zips
for zip_file in ['Lending_default_train.zip', 'Lending_default_holdout.zip']:
    if os.path.exists(zip_file):
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall()
        print(f'✓ Extracted {zip_file}')

# Load raw data
print('\nLoading raw data...')
df_train_tx = pd.read_csv('Lending_default_train_tx.csv')
df_train_acc = pd.read_csv('Lending_default_train_account.csv')
df_train_label = pd.read_csv('Lending_default_train_label.csv')
df_holdout_tx = pd.read_csv('Lending_default_holdout_tx.csv')
df_holdout_acc = pd.read_csv('Lending_default_holdout_account.csv')

# Clean unnamed columns
for df in [df_train_tx, df_train_acc, df_train_label, df_holdout_tx, df_holdout_acc]:
    cols = [c for c in df.columns if c.startswith('Unnamed')]
    df.drop(columns=cols, inplace=True)

print(f'Training TX: {df_train_tx.shape}')
print(f'Training ACC: {df_train_acc.shape}')
print(f'Training Label: {df_train_label.shape}')
print(f'Holdout TX: {df_holdout_tx.shape}')
print(f'Holdout ACC: {df_holdout_acc.shape}')
print(f'\n✓ Raw data loaded')

# Step 2: Feature Engineering - Rolling Time-Series Features

In [ ]:
def engineer_rolling_features(tx_data, snapshot_date_str):
    """
    Create rolling time-series features by Restaurant_ID and Tx_date.
    
    Parameters:
    - tx_data: Transaction dataframe with Restaurant_ID, Tx_date, processing_volume, Tx_hours
    - snapshot_date_str: End date for feature calculation (e.g., '2022-02-01')
    
    Returns:
    - df_features: Flattened dataframe with rolling features by Restaurant_ID
    """
    
    # Parse snapshot date
    snapshot_date = pd.to_datetime(snapshot_date_str)
    
    # Define lookback windows (in days)
    windows = [7, 30, 120, 180]
    
    # Initialize results dataframe
    df_features = None
    
    # For each restaurant
    for window_days in windows:
        # Calculate cutoff date
        cutoff_date = snapshot_date - timedelta(days=window_days)
        
        # Filter data within window
        window_data = tx_data[
            (tx_data['Tx_date'] >= str(cutoff_date.date())) & 
            (tx_data['Tx_date'] <= snapshot_date_str)
        ].copy()
        
        # Aggregate by Restaurant_ID
        agg_dict = {
            'processing_volume': ['mean', 'min', 'max', 'std'],
            'Tx_hours': ['mean', 'min', 'max', 'std']
        }
        
        window_features = window_data.groupby('Restaurant_ID').agg(agg_dict).reset_index()
        
        # Flatten column names
        window_features.columns = ['Restaurant_ID',
                                   f'avg_proc_vol_{window_days}d',
                                   f'min_proc_vol_{window_days}d',
                                   f'max_proc_vol_{window_days}d',
                                   f'std_proc_vol_{window_days}d',
                                   f'avg_tx_hours_{window_days}d',
                                   f'min_tx_hours_{window_days}d',
                                   f'max_tx_hours_{window_days}d',
                                   f'std_tx_hours_{window_days}d']
        
        # Calculate coefficient of variation
        window_features[f'cv_proc_vol_{window_days}d'] = (
            window_features[f'std_proc_vol_{window_days}d'] / 
            window_features[f'avg_proc_vol_{window_days}d']
        ).fillna(0)
        
        window_features[f'cv_tx_hours_{window_days}d'] = (
            window_features[f'std_tx_hours_{window_days}d'] / 
            window_features[f'avg_tx_hours_{window_days}d']
        ).fillna(0)
        
        # Drop std columns (use CV instead)
        window_features.drop(
            columns=[f'std_proc_vol_{window_days}d', f'std_tx_hours_{window_days}d'],
            inplace=True
        )
        
        # Merge with previous features
        if df_features is None:
            df_features = window_features
        else:
            df_features = df_features.merge(window_features, on='Restaurant_ID', how='outer')
    
    print(f'Rolling features created: {df_features.shape}')
    print(f'Features per window: {len([c for c in df_features.columns if "7d" in c])}')
    
    return df_features

print('Feature engineering function defined')

In [ ]:
def add_temporal_features(tx_data, features_df):
    """
    Add snapshot date features extracted from Tx_date.
    
    Parameters:
    - tx_data: Transaction dataframe with Tx_date
    - features_df: Features dataframe to add temporal features to
    
    Returns:
    - features_df: Updated with temporal features
    """
    
    # Get latest date per restaurant
    latest_dates = tx_data.groupby('Restaurant_ID')['Tx_date'].max().reset_index()
    latest_dates.rename(columns={'Tx_date': 'snapshot_date'}, inplace=True)
    latest_dates['snapshot_date'] = pd.to_datetime(latest_dates['snapshot_date'])
    
    # Extract temporal features
    latest_dates['snapshot_day_of_week'] = latest_dates['snapshot_date'].dt.dayofweek + 1  # 1-7
    latest_dates['snapshot_day_of_year'] = latest_dates['snapshot_date'].dt.dayofyear
    latest_dates['snapshot_month'] = latest_dates['snapshot_date'].dt.month
    latest_dates['snapshot_quarter'] = latest_dates['snapshot_date'].dt.quarter
    
    # Merge with features
    features_df = features_df.merge(
        latest_dates[['Restaurant_ID', 'snapshot_day_of_week', 'snapshot_day_of_year', 
                      'snapshot_month', 'snapshot_quarter']],
        on='Restaurant_ID',
        how='left'
    )
    
    print(f'Temporal features added: {features_df.shape}')
    print(f'Temporal columns: snapshot_day_of_week, snapshot_day_of_year, snapshot_month, snapshot_quarter')
    
    return features_df

print('Temporal feature function defined')

In [ ]:
# Create training features
print('='*70)
print('FEATURE ENGINEERING - TRAINING DATA')
print('='*70)
print(f'\nSnapshot date: {df_train_tx["Tx_date"].max()}')

df_train_rolling = engineer_rolling_features(df_train_tx, df_train_tx['Tx_date'].max())
df_train_features = add_temporal_features(df_train_tx, df_train_rolling)

print(f'\nTraining features shape: {df_train_features.shape}')
print(f'\nFeature columns ({df_train_features.shape[1]}):')
for col in df_train_features.columns:
    print(f'  - {col}')

In [ ]:
# Create holdout features
print('\n' + '='*70)
print('FEATURE ENGINEERING - HOLDOUT DATA')
print('='*70)
print(f'\nSnapshot date: {df_holdout_tx["Tx_date"].max()}')

df_holdout_rolling = engineer_rolling_features(df_holdout_tx, df_holdout_tx['Tx_date'].max())
df_holdout_features = add_temporal_features(df_holdout_tx, df_holdout_rolling)

print(f'\nHoldout features shape: {df_holdout_features.shape}')

# Step 3: Feature Validation

In [ ]:
print('='*70)
print('FEATURE VALIDATION')
print('='*70)

print(f'\nTraining Features:')
print(f'  Shape: {df_train_features.shape}')
print(f'  Missing values: {df_train_features.isnull().sum().sum()}')
print(f'  Data types: {df_train_features.dtypes.value_counts().to_dict()}')

print(f'\nHoldout Features:')
print(f'  Shape: {df_holdout_features.shape}')
print(f'  Missing values: {df_holdout_features.isnull().sum().sum()}')

print(f'\nFeature Statistics (Training):')  
print(df_train_features.describe().round(3).to_string())

print(f'\n✓ Feature validation complete')

# Step 4: Merge with Account Data and Target

In [ ]:
# Merge training data
print('Merging training data...')
df_train_final = df_train_features.merge(df_train_acc, on='Restaurant_ID', how='inner')
df_train_final = df_train_final.merge(df_train_label, on='Restaurant_ID', how='inner')

# Merge holdout data
print('Merging holdout data...')
df_holdout_final = df_holdout_features.merge(df_holdout_acc, on='Restaurant_ID', how='inner')

print(f'\nTraining final: {df_train_final.shape}')
print(f'Holdout final: {df_holdout_final.shape}')
print(f'\nEvent rate: {df_train_final["loan_default"].mean()*100:.2f}%')
print(f'✓ Data merged')

# Step 5: 80/20 Train-Test Split

In [ ]:
# Prepare
y = df_train_final['loan_default']
X = df_train_final.drop(columns=['loan_default', 'Restaurant_ID'])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Create dfs
df_train_set = X_train.copy()
df_train_set['loan_default'] = y_train.values
df_test_set = X_test.copy()
df_test_set['loan_default'] = y_test.values

print('='*70)
print('TRAIN-TEST SPLIT')
print('='*70)
print(f'Training: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)')
print(f'\nEvent rates:')
print(f'  Train: {y_train.mean()*100:.2f}%')
print(f'  Test: {y_test.mean()*100:.2f}%')
print(f'\nTotal features: {X_train.shape[1]}')

# Step 6: Train Models

In [ ]:
# Logistic Regression
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)

df_train_set['pred_lr'] = lr.predict_proba(X_train)[:, 1]
df_test_set['pred_lr'] = lr.predict_proba(X_test)[:, 1]

auc_train_lr = roc_auc_score(y_train, df_train_set['pred_lr'])
auc_test_lr = roc_auc_score(y_test, df_test_set['pred_lr'])
print(f'  Train AUC: {auc_train_lr:.4f}, Test AUC: {auc_test_lr:.4f}')

In [ ]:
# Gradient Boosting
print('Training Gradient Boosting...')
gb = GradientBoostingClassifier(
    n_estimators=100, max_leaf_nodes=10, learning_rate=0.1,
    min_samples_leaf=0.05, random_state=42
)
gb.fit(X_train, y_train)

df_train_set['pred_gb'] = gb.predict_proba(X_train)[:, 1]
df_test_set['pred_gb'] = gb.predict_proba(X_test)[:, 1]

auc_train_gb = roc_auc_score(y_train, df_train_set['pred_gb'])
auc_test_gb = roc_auc_score(y_test, df_test_set['pred_gb'])
print(f'  Train AUC: {auc_train_gb:.4f}, Test AUC: {auc_test_gb:.4f}')

# Step 7: Model Comparison & Selection

In [ ]:
print('='*70)
print('MODEL COMPARISON')
print('='*70)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Gradient Boosting'],
    'Train AUC': [auc_train_lr, auc_train_gb],
    'Test AUC': [auc_test_lr, auc_test_gb],
    'Overfit Gap': [abs(auc_train_lr - auc_test_lr), abs(auc_train_gb - auc_test_gb)]
})
print('\n' + comparison.round(4).to_string(index=False))

best_idx = 0 if auc_test_lr > auc_test_gb else 1
best_name = ['Logistic Regression', 'Gradient Boosting'][best_idx]
best_model = [lr, gb][best_idx]
best_pred_col = ['pred_lr', 'pred_gb'][best_idx]
best_auc = max(auc_test_lr, auc_test_gb)

print(f'\n✓ Best Model: {best_name} (Test AUC: {best_auc:.4f})')

# Step 8: Performance Metrics

In [ ]:
def show_metrics(y_true, y_pred_prob, name='Test'):
    y_pred = (y_pred_prob > 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    
    print(f'\n{name} Set Metrics:')
    print(f'  Accuracy: {accuracy_score(y_true, y_pred):.4f}')
    print(f'  Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}')
    print(f'  Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'  Recall: {recall_score(y_true, y_pred):.4f}')
    print(f'  ROC-AUC: {roc_auc_score(y_true, y_pred_prob):.4f}')
    print(f'  Confusion Matrix:')
    print(f'    TN: {cm[0,0]:,}  FP: {cm[0,1]:,}')
    print(f'    FN: {cm[1,0]:,}  TP: {cm[1,1]:,}')

print('='*70)
print(f'{best_name.upper()} PERFORMANCE')
print('='*70)
show_metrics(y_train, df_train_set[best_pred_col], 'Train')
show_metrics(y_test, df_test_set[best_pred_col], 'Test')

# Step 9: Time Series - Actual vs Predicted by Period

In [ ]:
def plot_calibration(df_data, pred_col, actual_col, title='Test', n_periods=10):
    df_temp = df_data.copy()
    df_temp['period'] = pd.qcut(range(len(df_temp)), q=n_periods,
                                 labels=[f'P{i+1}' for i in range(n_periods)])
    
    ts = df_temp.groupby('period').agg({
        actual_col: 'mean',
        pred_col: 'mean'
    })
    
    ts['actual_pct'] = ts[actual_col] * 100
    ts['pred_pct'] = ts[pred_col] * 100
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    x = range(len(ts))
    
    ax.plot(x, ts['actual_pct'], marker='o', linewidth=2.5, markersize=8,
            label='Actual Default Rate', color='#d62728')
    ax.plot(x, ts['pred_pct'], marker='s', linewidth=2.5, markersize=8,
            label='Predicted Default Rate', color='#1f77b4')
    
    ax.set_xlabel('Period', fontsize=12, fontweight='bold')
    ax.set_ylabel('Default Rate (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'{title}: Actual vs Predicted Default Rates', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(ts.index)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Metrics
    mae = abs(ts['actual_pct'] - ts['pred_pct']).mean()
    print(f'{title} Calibration: MAE = {mae:.2f}%')
    print(ts[['actual_pct', 'pred_pct']].round(2).to_string())

print('='*70)
print('TRAINING DATA: Actual vs Predicted')
print('='*70)
plot_calibration(df_train_set, best_pred_col, 'loan_default', 'Training')

In [ ]:
print('\n' + '='*70)
print('TEST DATA: Actual vs Predicted')
print('='*70)
plot_calibration(df_test_set, best_pred_col, 'loan_default', 'Test')

# Step 10: Score Holdout Data

In [ ]:
# Prepare holdout
X_holdout = df_holdout_final.drop(columns=[c for c in df_holdout_final.columns if c in ['Restaurant_ID']])

# Align features
for col in X_train.columns:
    if col not in X_holdout.columns:
        X_holdout[col] = 0
X_holdout = X_holdout[X_train.columns]

# Score
holdc_pred = best_model.predict_proba(X_holdout)[:, 1]

# Results
results = pd.DataFrame({
    'Restaurant_ID': df_holdout_final['Restaurant_ID'].values,
    'Predicted_Default_Probability': np.round(holdout_pred, 4),
    'Predicted_Default_Score_0_100': np.round(holdout_pred * 100, 2)
})

# Save
results.to_csv('holdout_predictions_with_fe.csv', index=False)

print('='*70)
print(f'HOLDOUT SCORING - {best_name}')
print('='*70)
print(f'Total: {len(results):,} records')
print(f'\nPrediction Distribution:')
print(results['Predicted_Default_Probability'].describe().round(4))
print(f'\nFirst 10:')
print(results.head(10).to_string(index=False))
print(f'\n✓ Saved to holdout_predictions_with_fe.csv')

# Step 11: Feature Importance Analysis

In [ ]:
# Feature importance from Gradient Boosting
if best_idx == 1:  # If GB is best
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print('='*70)
    print('FEATURE IMPORTANCE - TOP 20')
    print('='*70)
    print(feature_importance.head(20).round(4).to_string(index=False))
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 8))
    feature_importance.head(20).sort_values('importance').plot(
        kind='barh', x='feature', y='importance', ax=ax, legend=False
    )
    ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
    ax.set_title('Top 20 Most Important Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Step 12: Summary

In [ ]:
print('='*70)
print('FINAL SUMMARY')
print('='*70)
print(f'\nData:')
print(f'  Training: {len(X_train):,}')
print(f'  Test: {len(X_test):,}')
print(f'  Holdout: {len(results):,}')
print(f'\nFeatures Engineered:')
print(f'  Rolling windows: 7d, 30d, 120d, 180d')
print(f'  Metrics per window: avg, min, max, cv (4 windows × 2 metrics × 4 = 32 features)')
print(f'  Temporal features: day_of_week, day_of_year, month, quarter (4 features)')
print(f'  Account features: ownership_type, restaurant_category, market_segment')
print(f'  Total features used: {X_train.shape[1]}')
print(f'\nBest Model: {best_name}')
print(f'  Test AUC: {best_auc:.4f}')
print(f'\nOutput:')
print(f'  ✓ holdout_predictions_with_fe.csv')
print(f'\n✓ Complete!')